In [6]:
import sys
sys.path.append("..")
import ee, geemap
import pandas as pd
from src.config import GEE_PROJECT, AOI_BOX

ee.Initialize(project=GEE_PROJECT)
aoi = ee.Geometry.Rectangle(AOI_BOX)

# 1 km grid: har cell ek feature
def make_grid(aoi, cell_deg=0.01):          # 0.01 degree ~ 1.1 km
    xmin, ymin, xmax, ymax = AOI_BOX
    feats = []
    x = xmin
    i = 0
    while x < xmax - 1e-9:
        y = ymin
        while y < ymax - 1e-9:
            geom = ee.Geometry.Rectangle([x, y, x + cell_deg, y + cell_deg])
            feats.append(ee.Feature(geom, {"cell_id": i, "lon": x + cell_deg/2, "lat": y + cell_deg/2}))
            i += 1
            y += cell_deg
        x += cell_deg
    return ee.FeatureCollection(feats)

grid = make_grid(aoi)
print("Cells:", grid.size().getInfo())

Cells: 5600


In [7]:
srtm = ee.Image("USGS/SRTMGL1_003")
elev = srtm.rename("elevation")
slope = ee.Terrain.slope(srtm).rename("slope")

# permanent paani (nadi/samudra) se doori
water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(80)
dist_water = (water.selfMask().fastDistanceTransform(256).sqrt()
              .multiply(ee.Image.pixelArea().sqrt()).rename("dist_water_m"))

pop = pop = (ee.ImageCollection("WorldPop/GP/100m/R2025A/v1/pop")
       .filter(ee.Filter.eq("country", "IND"))
       .filter(ee.Filter.eq("year", 2020)).mosaic().rename("population"))

static = elev.addBands([slope, dist_water, pop])

In [8]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

def rain_sum(start, end, name):
    return chirps.filterDate(start, end).sum().rename(name)

# flood 16 Aug ko shuru hua, isliye us se pehle ka rainfall
rain3 = rain_sum("2022-08-13", "2022-08-16", "rain_3d")
rain7 = rain_sum("2022-08-09", "2022-08-16", "rain_7d")

features_img = static.addBands([rain3, rain7])

In [4]:
# 5600 cells ek saath bhaari ho sakta hai, isliye tukdon mein
def extract(grid_fc, img, scale=100):
    return img.reduceRegions(
        collection=grid_fc,
        reducer=ee.Reducer.mean(),
        scale=scale,
        tileScale=4,
    )

features_fc = extract(grid, features_img)

# pehle sirf 5 cell dekho, check karne ke liye
sample = features_fc.limit(5).getInfo()
for f in sample["features"]:
    print(f["properties"])

{'cell_id': 0, 'dist_water_m': 5848.627943665501, 'elevation': 11.290140845070427, 'lat': 20.205, 'lon': 86.205, 'population': 6.796448824621125, 'rain_3d': 109.66996573888721, 'rain_7d': 296.69548374881754, 'slope': 0.8753489450102196}
{'cell_id': 1, 'dist_water_m': 6632.403664775478, 'elevation': 11.392785951589941, 'lat': 20.215, 'lon': 86.205, 'population': 8.180341422128668, 'rain_3d': 109.89700584176262, 'rain_7d': 296.544808373363, 'slope': 0.8696580005486476}
{'cell_id': 2, 'dist_water_m': 7441.248706293106, 'elevation': 11.541351479664508, 'lat': 20.225, 'lon': 86.205, 'population': 7.426487340053502, 'rain_3d': 109.89693307902245, 'rain_7d': 296.54481054210333, 'slope': 0.880841299007967}
{'cell_id': 3, 'dist_water_m': 8271.114299215365, 'elevation': 12.838443284290461, 'lat': 20.235000000000003, 'lon': 86.205, 'population': 9.108728841361193, 'rain_3d': 109.8970058417626, 'rain_7d': 296.5448083733629, 'slope': 0.9694940370141191}
{'cell_id': 4, 'dist_water_m': 9150.251034679

In [9]:
flood_mask = flood_clean.unmask(0).rename("flood")
labels_fc = flood_mask.reduceRegions(
    collection=features_fc, reducer=ee.Reducer.mean(), scale=30, tileScale=4
)
# flood_frac: cell ka kitna hissa doob gaya (0 se 1)

NameError: name 'flood_clean' is not defined

In [10]:
def s1_collection(start, end, orbit):
    return (ee.ImageCollection("COPERNICUS/S1_GRD")
            .filterBounds(aoi).filterDate(start, end)
            .filter(ee.Filter.eq("instrumentMode", "IW"))
            .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
            .filter(ee.Filter.eq("orbitProperties_pass", orbit))
            .select("VV"))

def flood_map(before, after, orbit, diff_thr=-3, water_thr=-15):
    b = s1_collection(*before, orbit).mean().focal_median(30, "circle", "meters")
    a = s1_collection(*after, orbit).mean().focal_median(30, "circle", "meters")
    flood = (a.subtract(b)).lt(diff_thr).And(a.lt(water_thr))
    perm = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50)
    flood = flood.updateMask(perm.Not())
    slope_ = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
    return flood.updateMask(slope_.lt(5)).selfMask().clip(aoi)

def event_flood(before, after):
    fd = flood_map(before, after, "DESCENDING")
    fa = flood_map(before, after, "ASCENDING")
    merged = ee.ImageCollection([fd.unmask(0), fa.unmask(0)]).max().selfMask()
    conn = merged.connectedPixelCount(maxSize=100, eightConnected=True)
    return merged.updateMask(conn.gte(8))

flood_clean = event_flood(("2022-08-04", "2022-08-17"), ("2022-08-18", "2022-08-24"))

In [11]:
rows = labels_fc.limit(5).getInfo()
for f in rows["features"]:
    print(f["properties"]["cell_id"], f["properties"]["mean"])

NameError: name 'labels_fc' is not defined

In [12]:
import pandas as pd

def to_df(fc):
    feats = fc.getInfo()["features"]
    return pd.DataFrame([f["properties"] for f in feats])

df = to_df(labels_fc)
df = df.rename(columns={"mean": "flood_frac"})

print("Total cells:", len(df))
print("Cells with any flood (>0):", (df.flood_frac > 0).sum())
print("Cells >= 10% flooded:", (df.flood_frac >= 0.10).sum())
print("Cells >= 30% flooded:", (df.flood_frac >= 0.30).sum())
print(df.flood_frac.describe())

NameError: name 'labels_fc' is not defined

In [13]:
import pandas as pd

def to_df(fc, chunk=2500):
    n = fc.size().getInfo()
    parts = []
    for start in range(0, n, chunk):
        sub = ee.FeatureCollection(fc.toList(chunk, start))
        feats = sub.getInfo()["features"]
        parts.append(pd.DataFrame([f["properties"] for f in feats]))
        print(f"fetched {start + len(feats)} / {n}")
    return pd.concat(parts, ignore_index=True)

# labels_fc me flood ka 'mean' hai, features_fc me baaki features hain
feat_df = to_df(features_fc)
lab_df = to_df(labels_fc)[["cell_id", "mean"]].rename(columns={"mean": "flood_frac"})

df = feat_df.merge(lab_df, on="cell_id")
print("Total cells:", len(df))
print("Cells with any flood (>0):", (df.flood_frac > 0).sum())
print("Cells >= 10% flooded:", (df.flood_frac >= 0.10).sum())
print("Cells >= 30% flooded:", (df.flood_frac >= 0.30).sum())
print(df.flood_frac.describe())

df.to_csv("../data/processed/features_2022.csv", index=False)
print("Saved: data/processed/features_2022.csv")

fetched 2500 / 5600
fetched 5000 / 5600
fetched 5600 / 5600


NameError: name 'labels_fc' is not defined

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003")
hand = ee.Image("MERIT/Hydro/v1_0_1").select("hnd").rename("hand")      # nadi se upar ki unchai
elev_mean = srtm.focal_mean(1000, "circle", "meters").rename("elev_mean_1km")
elev_rel = srtm.subtract(elev_mean).rename("elev_rel")                   # aas-paas se kitna uncha/neecha

wc = ee.Image("ESA/WorldCover/v200").select("Map")
cropland = wc.eq(40).rename("cropland")
builtup = wc.eq(50).rename("builtup")
wetland = wc.eq(90).Or(wc.eq(95)).rename("wetland_mangrove")

new_img = hand.addBands([elev_mean, elev_rel, cropland, builtup, wetland])

new_fc = new_img.reduceRegions(
    collection=grid, reducer=ee.Reducer.mean(), scale=100, tileScale=4
)
new_df = to_df(new_fc)            # wahi chunk wala function jo pehle banaya tha
new_df = new_df.drop(columns=["lat", "lon"], errors="ignore")
print(new_df.describe().round(2))

NameError: name 'to_df' is not defined